# Re download data

In [1]:
pip install requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\lebel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import os
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
import time
import socket
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, Image as IPImage, HTML
import time



_last_check_time = 0
_last_status = True

# -------------------------------
# Config
# -------------------------------
DATA_DIR = "D:/multimodal_pipeline/data"
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test_public.tsv")
}
OUTPUT_DIRS = {
    "train": os.path.join(DATA_DIR, "train_images"),
    "validate": os.path.join(DATA_DIR, "validate_images"),
    "test": os.path.join(DATA_DIR, "test_images")
}
FAILED_LOG_DIR = os.path.join(DATA_DIR, "failed_downloads")
os.makedirs(FAILED_LOG_DIR, exist_ok=True)

for key, path in OUTPUT_DIRS.items():
    os.makedirs(path, exist_ok=True)

MAX_THREADS = 8
MAX_RETRIES = 3
VISUAL_SAMPLE_EVERY = 100  # show one sample every 100 images

# -------------------------------
# Check Internet Connection
# -------------------------------
def internet_on(timeout=5):
    """
    Check internet connectivity using an HTTP request (more reliable on Windows).
    """
    test_urls = ["https://www.google.com", "https://pypi.org", "https://huggingface.co"]
    for url in test_urls:
        try:
            response = requests.get(url, timeout=timeout)
            if response.status_code == 200:
                return True
        except requests.ConnectionError:
            continue
        except Exception:
            continue
    return False

def internet_on_cached(timeout=5, check_interval=15):
    global _last_check_time, _last_status
    now = time.time()
    if now - _last_check_time < check_interval:
        return _last_status
    _last_status = internet_on(timeout)
    _last_check_time = now
    return _last_status

# -------------------------------
# Display random sample in notebook
# -------------------------------
def show_image_sample(image_path, text):
    display(HTML(f"<b>{text}</b>"))
    display(IPImage(filename=image_path))

# -------------------------------
# Single Image Download
# -------------------------------
def download_image(row, output_dir, index=None):
    img_id = row["id"]
    url = row["image_url"]
    save_path = os.path.join(output_dir, f"{img_id}.jpg")

    # Skip if already exists or URL invalid
    if os.path.exists(save_path) or not isinstance(url, str) or not url.startswith("http"):
        return "skipped", None

    # Wait for internet
    while not internet_on_cached():
        print("No internet connection. Waiting 10 seconds...")
        time.sleep(10)

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content)).convert("RGB")
            img.save(save_path)

            # Visualize every VISUAL_SAMPLE_EVERY image
            if index is not None and index % VISUAL_SAMPLE_EVERY == 0:
                show_image_sample(save_path, row.get("title", "No title"))

            return "downloaded", None
        else:
            return "failed", row
    except Exception:
        return "failed", row

# -------------------------------
# Parallel Download with Progress
# -------------------------------
def download_images_parallel(df, output_dir):
    failed_records = []
    skipped_count = 0
    downloaded_count = 0
    failed_count = 0

    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {executor.submit(download_image, row, output_dir, idx): idx for idx, row in df.iterrows()}
        for future in tqdm(as_completed(futures), total=len(futures)):
            status, failed_row = future.result()
            if status == "skipped":
                skipped_count += 1
            elif status == "downloaded":
                downloaded_count += 1
            elif status == "failed":
                failed_count += 1
                failed_records.append(failed_row)

    return downloaded_count, skipped_count, failed_records

# -------------------------------
# Retry Failed Downloads
# -------------------------------
def retry_failed_downloads(failed_records, output_dir):
    for attempt in range(1, MAX_RETRIES + 1):
        if not failed_records:
            break
        print(f"\nRetry attempt {attempt} for {len(failed_records)} failed downloads...")
        df_failed = pd.DataFrame(failed_records)
        _, _, failed_records = download_images_parallel(df_failed, output_dir)
    return failed_records

# -------------------------------
# Main Download Function
# -------------------------------
def download_split(split):
    df = pd.read_csv(TSV_FILES[split], sep="\t")
    output_dir = OUTPUT_DIRS[split]
    print(f"Processing {len(df)} images from {split} split...")

    # --- Preload downloaded image IDs ---
    existing_images = {
        os.path.splitext(f)[0]
        for f in os.listdir(output_dir)
        if f.lower().endswith(".jpg")
    }
    print(f"Found {len(existing_images)} already downloaded images in {split} folder.")

    # --- Filter the DataFrame ---
    df_to_download = df[~df["id"].astype(str).isin(existing_images)]
    print(f"Remaining images to download: {len(df_to_download)}")

    if len(df_to_download) == 0:
        print(f"✅ All images for {split} are already downloaded.")
        return

    downloaded, skipped, failed_records = download_images_parallel(df, OUTPUT_DIRS[split])
    print(f"\nInitial pass finished: Downloaded={downloaded}, Skipped={skipped}, Failed={len(failed_records)}")

    failed_records = retry_failed_downloads(failed_records, OUTPUT_DIRS[split])
    print(f"\nFinal failed downloads for {split}: {len(failed_records)}")

    if failed_records:
        failed_df = pd.DataFrame(failed_records)
        failed_file = os.path.join(FAILED_LOG_DIR, f"failed_{split}.csv")
        failed_df.to_csv(failed_file, index=False)
        print(f"Logged failed downloads to {failed_file}")

# -------------------------------
# Run for all splits
# -------------------------------
for split in ["train", "validate", "test"]:
    download_split(split)


Processing 564000 images from train split...
Found 380981 already downloaded images in train folder.
Remaining images to download: 183019


In [ ]:
import os
import glob
import time
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display, Image as IPImage, HTML

# ======================================================
# CONFIGURATION
# ======================================================
DATA_DIR = "D:/multimodal_pipeline/data"
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test_public.tsv")
}
OUTPUT_DIRS = {
    "train": os.path.join(DATA_DIR, "train_images"),
    "validate": os.path.join(DATA_DIR, "validate_images"),
    "test": os.path.join(DATA_DIR, "test_images")
}
FAILED_LOG_DIR = os.path.join(DATA_DIR, "failed_downloads")
os.makedirs(FAILED_LOG_DIR, exist_ok=True)
for key, path in OUTPUT_DIRS.items():
    os.makedirs(path, exist_ok=True)

MAX_THREADS = 8
MAX_RETRIES = 3
VISUAL_SAMPLE_EVERY = 100

# ======================================================
# INTERNET CHECK
# ======================================================
_last_check_time = 0
_last_status = True

def internet_on(timeout=5):
    test_urls = ["https://www.google.com", "https://pypi.org", "https://huggingface.co"]
    for url in test_urls:
        try:
            response = requests.get(url, timeout=timeout)
            if response.status_code == 200:
                return True
        except requests.ConnectionError:
            continue
        except Exception:
            continue
    return False

def internet_on_cached(timeout=5, check_interval=15):
    global _last_check_time, _last_status
    now = time.time()
    if now - _last_check_time < check_interval:
        return _last_status
    _last_status = internet_on(timeout)
    _last_check_time = now
    return _last_status

# ======================================================
# VISUALIZATION (OPTIONAL)
# ======================================================
def show_image_sample(image_path, text):
    try:
        display(HTML(f"<b>{text}</b>"))
        display(IPImage(filename=image_path))
    except Exception:
        pass

# ======================================================
# IMAGE DOWNLOAD
# ======================================================
def download_image(row, output_dir, index=None):
    img_id = str(row["id"])
    url = row["image_url"]
    save_path = os.path.join(output_dir, f"{img_id}.jpg")

    if os.path.exists(save_path) or not isinstance(url, str) or not url.startswith("http"):
        return "skipped", None

    while not internet_on_cached():
        print("⚠️ No internet connection. Retrying in 10 seconds...")
        time.sleep(10)

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            if img.mode == "P":
                img = img.convert("RGBA")
            else:
                img = img.convert("RGB")
            img.save(save_path)

            if index is not None and index % VISUAL_SAMPLE_EVERY == 0:
                show_image_sample(save_path, row.get("title", "No title"))

            return "downloaded", None
        else:
            return "failed", row
    except Exception:
        return "failed", row

# ======================================================
# PARALLEL DOWNLOAD
# ======================================================
def download_images_parallel(df, output_dir):
    failed_records = []
    skipped_count = downloaded_count = failed_count = 0

    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {executor.submit(download_image, row, output_dir, idx): idx for idx, row in df.iterrows()}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading images"):
            status, failed_row = future.result()
            if status == "skipped":
                skipped_count += 1
            elif status == "downloaded":
                downloaded_count += 1
            elif status == "failed":
                failed_count += 1
                failed_records.append(failed_row)

    return downloaded_count, skipped_count, failed_records

# ======================================================
# RETRY FAILED DOWNLOADS
# ======================================================
def retry_failed_downloads(failed_records, output_dir):
    for attempt in range(1, MAX_RETRIES + 1):
        if not failed_records:
            break
        print(f"\n🔁 Retry attempt {attempt} for {len(failed_records)} failed downloads...")
        df_failed = pd.DataFrame(failed_records)
        _, _, failed_records = download_images_parallel(df_failed, output_dir)
    return failed_records

# ======================================================
# MAIN FUNCTION WITH SAFE TSV READER
# ======================================================
def download_split(split):
    print(f"\n📦 Loading {split} split...")

    # Try safe reading
    try:
        df = pd.read_csv(
            TSV_FILES[split],
            sep="\t",
            usecols=["id", "image_url", "title"],
            engine="python",
            on_bad_lines="skip"
        )
    except Exception as e:
        print(f"⚠️ Failed to read TSV for {split}: {e}")
        # fallback: try without usecols
        df = pd.read_csv(TSV_FILES[split], sep="\t", engine="python", on_bad_lines="skip")
        print("⚙️ Columns loaded:", df.columns.tolist())

    output_dir = OUTPUT_DIRS[split]
    print(f"Processing {len(df):,} records from '{split}'...")

    # Fast resume check
    existing_files = glob.glob(os.path.join(output_dir, "*.jpg"))
    existing_images = set(os.path.splitext(os.path.basename(f))[0] for f in existing_files)
    print(f"🗂️ Found {len(existing_images):,} already downloaded images.")

    df["id"] = df["id"].astype(str)
    df_to_download = df[~df["id"].isin(existing_images)]
    print(f"🕐 Remaining images to download: {len(df_to_download):,}")

    if df_to_download.empty:
        print(f"✅ All images for '{split}' are already downloaded.")
        return

    downloaded, skipped, failed_records = download_images_parallel(df_to_download, output_dir)
    print(f"\n✅ Pass finished: Downloaded={downloaded}, Skipped={skipped}, Failed={len(failed_records)}")

    failed_records = retry_failed_downloads(failed_records, output_dir)
    print(f"\n❌ Final failed downloads for {split}: {len(failed_records)}")

    if failed_records:
        failed_df = pd.DataFrame(failed_records)
        failed_file = os.path.join(FAILED_LOG_DIR, f"failed_{split}.csv")
        failed_df.to_csv(failed_file, index=False)
        print(f"📝 Logged failed downloads to {failed_file}")

# ======================================================
# EXECUTION
# ======================================================
for split in ["train", "validate", "test"]:
    download_split(split)

print("\n🎉 All splits processed successfully!")


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.